# Clustering methods

Aggregation has two halves: **grouping** similar periods, then **representing**
each group with one profile. This page is about the first half — the clustering
method, set with `ClusterConfig(method=...)`. (For the second, see
[Representations](representations.ipynb).)

tsam offers six methods:

| method | how it groups | notes |
|--------|---------------|-------|
| `hierarchical` | merges the closest periods step by step | the **default**; robust, deterministic |
| `kmeans` | minimises distance to cluster centroids | fast, often most accurate |
| `kmedoids` | like k-means but centers are **real** periods | exact (MILP), slower |
| `kmaxoids` | centers pushed toward extremes | preserves spread, not the average |
| `averaging` | splits the timeline into equal chunks | simplest baseline, ignores similarity |
| `contiguous` | only merges temporally adjacent periods | keeps calendar order |

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

import tsam

pio.renderers.default = "notebook_connected"

raw = pd.read_csv("testdata.csv", index_col=0, parse_dates=True)
data = raw.loc["2010-01-01":"2010-02-11"]  # six weeks of hourly data

## Accuracy and speed

Run all six at the same size and compare. The right choice trades accuracy
against runtime (which matters once you have thousands of periods):

In [ ]:
from tsam import ClusterConfig

methods = ["hierarchical", "kmeans", "kmedoids", "kmaxoids", "averaging", "contiguous"]
rows = {}
for m in methods:
    r = tsam.aggregate(
        data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method=m)
    )
    rows[m] = {
        "mean RMSE": round(float(r.accuracy.rmse.mean()), 4),
        "clustering time (s)": round(r.clustering_duration, 3),
    }
pd.DataFrame(rows).T.sort_values("mean RMSE")

## How the grouping plays out

Same data, same number of clusters — but the methods assign days to groups
differently, so the reconstructed series differ. Over one week: `kmeans` and
`hierarchical` track the original closely, while `averaging` (blind to
similarity) is visibly coarser.

In [ ]:
week = slice("2010-01-11", "2010-01-17")
frames = []
for m in ["hierarchical", "kmeans", "kmedoids", "averaging"]:
    r = tsam.aggregate(
        data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method=m)
    )
    s = r.reconstructed.loc[week, "Load"]
    frames.append(pd.DataFrame({"time": s.index, "Load": s.values, "method": m}))
orig = data.loc[week, "Load"]
frames.append(
    pd.DataFrame({"time": orig.index, "Load": orig.values, "method": "original"})
)
px.line(
    pd.concat(frames),
    x="time",
    y="Load",
    color="method",
    title="One week reconstructed, by clustering method",
)

## Medoids: real days, not averages

`kmeans`/`hierarchical` build representatives by averaging, which can smooth away
realistic shapes. `kmedoids` instead picks an **actual period** as each center, so
every typical day is a day that really happened — useful when downstream logic
needs physically consistent profiles:

In [ ]:
centroid = tsam.aggregate(
    data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method="kmeans")
)
medoid = tsam.aggregate(
    data, n_clusters=6, period_duration="1D", cluster=ClusterConfig(method="kmedoids")
)
medoid.plot.compare(
    columns=["Load"],
    mode="duration_curve",
    title="kmedoids reconstruction vs. original (duration curve)",
)

## Which to use

- **Start with `hierarchical`** (the default) or **`kmeans`** — both are fast and
  accurate for most data.
- Use **`kmedoids`** when representatives must be real, observed periods.
- Use **`kmaxoids`** when the spread / extremes matter more than the average.
- Use **`contiguous`** when periods must stay in calendar order.

The companion choice — how each cluster becomes a single profile — is covered in
[Representations](representations.ipynb).